# 06. Core data structures

## Learning goals
1. Explain how `list`, `dict` and `set` work inside CPython.
2. Implement linked lists, stacks, queues, heaps and hash maps.
3. Implement a BST, an AVL tree, a trie and union-find.
4. State and justify the cost of each operation.

In [ ]:
import sys, pathlib
for _p in (pathlib.Path.cwd(), pathlib.Path.cwd().parent):
    if (_p / "src" / "understanding_python").exists():
        sys.path.insert(0, str(_p / "src"))
print("python", sys.version.split()[0])

## list, dict and set internals
- **list** is a dynamic array of pointers. Index is O(1). `insert(0, x)` shifts n items: O(n). `append` is amortised O(1).
- **dict** is an open-addressing hash table with a compact entry array that keeps insertion order. Lookup is expected O(1).
- **set** is the same table without values.

The benchmark shows the gap between O(n) membership in a list and expected O(1) in a set.

In [ ]:
import random
from collections import deque
from understanding_python.bench import best_time
rng = random.Random(7)
n = 5000
data = rng.sample(range(10 * n), n)
as_list, as_set = data, set(data)
probes = [rng.randrange(10 * n) for _ in range(200)]
t_list = best_time(lambda: [p in as_list for p in probes], 3)
t_set = best_time(lambda: [p in as_set for p in probes], 3)
print(f"list {t_list:.4f}s set {t_set:.6f}s")
assert t_set * 5 < t_list
l = list(range(20000)); d = deque(range(20000))
t_l = best_time(lambda: [l.insert(0, 0) for _ in range(200)], 1)
t_d = best_time(lambda: [d.appendleft(0) for _ in range(200)], 1)
assert t_d < t_l       # deque pops and pushes at both ends in O(1)
assert {1: "a", 2: "b"} == {2: "b", 1: "a"} and list({3: 0, 1: 0}) == [3, 1]

## Linked list, stack, queue
A singly linked list pushes at the head in O(1) and searches in O(n). A stack gives LIFO order; a queue gives FIFO order.
Bracket matching uses a stack: each character is pushed or popped once, so the cost is O(n).

In [ ]:
from understanding_python.structures import LinkedList, Stack, Queue, balanced
ll = LinkedList()
for i in range(5): ll.push_front(i)
assert list(ll) == [4, 3, 2, 1, 0]
ll.reverse(); assert list(ll) == [0, 1, 2, 3, 4]
assert ll.remove(2) and not ll.remove(99) and len(ll) == 4
s = Stack(); [s.push(i) for i in range(3)]
assert [s.pop(), s.pop(), s.pop()] == [2, 1, 0]
q = Queue(); [q.enqueue(i) for i in range(3)]
assert [q.dequeue(), q.dequeue(), q.dequeue()] == [0, 1, 2]
assert balanced("{[()()]}") and not balanced("([)]") and not balanced("((")

## Binary heap
A heap stores a complete binary tree in a list: children of i sit at 2i+1 and 2i+2. Each parent is at most its children.
`push` sifts up and `pop` sifts down. Both follow one root-to-leaf path of length floor(log2 n), so each costs O(log n).
Building by n pushes costs O(n log n).

In [ ]:
import heapq
from understanding_python.structures import MinHeap
rng = random.Random(11)
vals = [rng.randint(0, 1000) for _ in range(500)]
h = MinHeap()
for v in vals: h.push(v)
assert all(h.a[(i - 1) // 2] <= h.a[i] for i in range(1, len(h.a)))   # heap property
out = [h.pop() for _ in range(len(h))]
assert out == sorted(vals)
ref = list(vals); heapq.heapify(ref)
assert [heapq.heappop(ref) for _ in range(len(ref))] == out

## Hash map
Hash the key, take it modulo the bucket count, scan a short chain. If the load factor stays below 0.75 the expected chain length is O(1).
Doubling the table when the load passes 0.75 keeps that bound; the resize cost amortises to O(1) per insert, as for lists.

In [ ]:
from understanding_python.structures import HashMap
rng = random.Random(3)
m, ref = HashMap(), {}
for _ in range(3000):
    k, v = rng.randrange(1000), rng.random()
    m.put(k, v); ref[k] = v
assert len(m) == len(ref) and all(m.get(k) == v for k, v in ref.items())
assert m.capacity >= len(m) / 0.75
assert m.delete(next(iter(ref))) and len(m) == len(ref) - 1
print("capacity", m.capacity, "entries", len(m))

## Binary search tree and AVL tree
A BST keeps left < node < right. Cost is O(h). Sorted inserts make a chain with h = n.

An AVL tree keeps |height(left) - height(right)| <= 1 using rotations.
**Height bound.** Let N(h) be the fewest nodes in an AVL tree of height h. N(h) = N(h-1) + N(h-2) + 1, which is Fibonacci-like,
so N(h) >= phi^h / sqrt(5) - 1 and h <= 1.44 log2(n + 2). Every operation is therefore O(log n).

In [ ]:
import math
from understanding_python.structures import BST, AVLTree
n = 1023
b, a = BST(), AVLTree()
for k in range(n):
    b.insert(k); a.insert(k)
print("BST height", b.height(), "AVL height", a.height())
assert b.height() == n
assert a.height() <= 1.44 * math.log2(n + 2) and a.is_balanced()
assert a.inorder() == b.inorder() == list(range(n))
assert 500 in a and 5000 not in a

## Trie
A trie stores one character per edge. Insert and lookup cost O(L) for a word of length L, independent of how many words exist.

In [ ]:
from understanding_python.structures import Trie
t = Trie()
for w in ["tea", "ten", "inn", "teapot", "tent"]: t.insert(w)
assert t.search("tea") and not t.search("te") and t.starts_with("te")
assert t.with_prefix("ten") == ["ten", "tent"]
assert t.with_prefix("tea") == ["tea", "teapot"] and t.with_prefix("z") == []

## Union-find
Disjoint sets with union by size and path compression. A tree of height h has at least 2^h nodes, so find is O(log n) even without compression.
With both heuristics, m operations cost O(m alpha(n)), where alpha is the inverse Ackermann function, below 5 for any practical n.

In [ ]:
from understanding_python.structures import UnionFind
uf = UnionFind(10)
for a_, b_ in [(0, 1), (2, 3), (1, 3), (5, 6), (7, 8), (8, 9)]:
    uf.union(a_, b_)
assert uf.find(0) == uf.find(3) and uf.find(0) != uf.find(5)
assert uf.components == 10 - 6 + 0 and not uf.union(0, 2)
big = UnionFind(100_000)
for i in range(99_999): big.union(i, i + 1)
assert big.components == 1 and big.find(0) == big.find(99_999)